# HotpotQA-VN: dữ liệu final → graph tiếng Việt → HippoRAG2

Dùng `queries.jsonl`, `corpus.jsonl`, `qrels.tsv` từ repo Prepare-data-HotpotQA-VN. Corpus có 9.822 tài liệu; chọn ít câu hỏi KHÔNG giảm corpus. Profile mặc định là pilot ablation 109 chunk chỉ giữ entity-relation và event-entity, không sinh event-event relation. Qwen3.5-2B local, embedding multilingual E5 trên CPU.

Mặc định `RUN_PHASE='prepare'` chỉ chuẩn bị dữ liệu và không cần GPU. Chọn GPU trước khi dùng `extract`, `build`, `benchmark` hoặc `all`; đổi phase lần lượt để checkpoint rõ ràng. Đọc `HOTPOTQA_VN.md` trước khi bắt đầu extraction toàn corpus.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
import requests
from google.colab import drive, files
drive.mount('/content/drive')
EXPERIMENT_PROFILE = 'no_event_pilot'  # baseline | ab_rp115 | ab_event_guard_v2 | ab_entity_event_v3 | full_pilot | no_event_pilot
PROFILES = {
    'baseline': {'directory': 'hotpotqa_vn_resumable_v2', 'chunks': 500, 'penalty': 1.05, 'event_relations': True},
    'ab_rp115': {'directory': 'hotpotqa_vn_ab_rp115', 'chunks': 109, 'penalty': 1.15, 'event_relations': True},
    'ab_event_guard_v2': {'directory': 'hotpotqa_vn_ab_event_guard_v2', 'chunks': 109, 'penalty': 1.15, 'event_relations': True},
    'ab_entity_event_v3': {'directory': 'hotpotqa_vn_ab_entity_event_v3', 'chunks': 109, 'penalty': 1.15, 'event_relations': False},
    'full_pilot': {'directory': 'hotpotqa_vn_full_pilot', 'chunks': 30, 'penalty': 1.15, 'event_relations': True, 'events': True, 'benchmark_variant': 'full', 'partial_build': True},
    'no_event_pilot': {'directory': 'hotpotqa_vn_no_event_pilot', 'chunks': 30, 'penalty': 1.15, 'event_relations': False, 'events': False, 'benchmark_variant': 'no_event', 'partial_build': True},
}
profile = PROFILES[EXPERIMENT_PROFILE]
RUN_ROOT = Path('/content/drive/MyDrive/AutoSchemaKG') / profile['directory']
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = RUN_ROOT / 'experiment'
OUTPUT_DIR = WORK_DIR / 'benchmark'
RUN_PHASE = 'prepare'  # prepare -> extract -> build -> package -> benchmark
MAX_QUESTIONS = 1000  # Changing this does NOT reduce the 9,822-document corpus
EXTRACTION_CHUNKS_PER_RUN = profile['chunks']
REPETITION_PENALTY = profile['penalty']
INCLUDE_EVENT_RELATIONS = profile['event_relations']
INCLUDE_EVENTS = profile.get('events', True)
BENCHMARK_VARIANT = profile.get('benchmark_variant', 'full')
ALLOW_PARTIAL_BUILD = profile.get('partial_build', False)
UPGRADE_CODE_FOR_RESUME = False  # Set True once only for a run created before resumable extraction
MODEL_ID = 'Qwen/Qwen3.5-2B'
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small'
PORT = 8000
CONTEXT_LENGTH = 4096
NEEDS_LLM = RUN_PHASE in {'extract', 'build', 'benchmark', 'all'}
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if NEEDS_LLM and gpu.returncode != 0:
    raise RuntimeError('This phase needs Qwen. Select Runtime > Change runtime type > GPU.')
print(gpu.stdout if gpu.returncode == 0 else 'CPU runtime: sufficient for prepare/package')
print('Experiment profile:', EXPERIMENT_PROFILE)
print('Persistent experiment directory:', RUN_ROOT)
print('Extraction chunks/run:', EXTRACTION_CHUNKS_PER_RUN)
print('Repetition penalty:', REPETITION_PENALTY)
print('Include event relations:', INCLUDE_EVENT_RELATIONS)
print('Include events:', INCLUDE_EVENTS)
print('Allow partial pilot build:', ALLOW_PARTIAL_BUILD)

## Clone code và ghim model/code revision

Các file VN phải được push lên GitHub trước khi dùng notebook này. Run mới giữ `UPGRADE_CODE_FOR_RESUME=False`. Nếu `RUN_ROOT` đã được tạo bằng bản cũ chưa hỗ trợ resume, đặt biến này thành `True` đúng một lần; notebook lưu commit cũ vào `code_upgrade_history` trước khi ghim commit mới.

In [ ]:
CODE_REF = 'experiment/no-event-small-model'
REPO_DIR = Path('/content/SmallScaledAutoSchemaKG_vn')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', CODE_REF, '--single-branch',
                    'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', str(REPO_DIR)], check=True)
revision_file = RUN_ROOT / 'revisions.json'
if revision_file.exists():
    revisions = json.loads(revision_file.read_text())
    if revisions['model'] != MODEL_ID or revisions['embedding'] != EMBEDDING_MODEL:
        raise ValueError('Model changed: use a NEW RUN_ROOT')
    status = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True).strip()
    if status:
        raise RuntimeError('Clone has local edits; preserve them before switching revision')
    if UPGRADE_CODE_FOR_RESUME:
        subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO_DIR, check=True)
        previous_commit = revisions['git_commit']
        revisions.setdefault('code_upgrade_history', []).append(previous_commit)
        revisions['git_commit'] = subprocess.check_output(
            ['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO_DIR, text=True).strip()
        revision_file.write_text(json.dumps(revisions, indent=2))
        print(f'Code upgraded for resume: {previous_commit} -> {revisions["git_commit"]}')
    subprocess.run(['git', 'checkout', '--detach', revisions['git_commit']], cwd=REPO_DIR, check=True)
else:
    # Also repair an older clone that was created from the repository default branch.
    subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
    def model_revision(model):
        response = requests.get(f'https://huggingface.co/api/models/{model}', timeout=30)
        response.raise_for_status()
        return response.json()['sha']
    revisions = {'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
                 'model': MODEL_ID, 'model_revision': model_revision(MODEL_ID),
                 'embedding': EMBEDDING_MODEL, 'embedding_revision': model_revision(EMBEDDING_MODEL)}
    if not (REPO_DIR / 'scripts/run_hotpotqa_vn.py').exists():
        raise RuntimeError('v2 files are not present on this GitHub revision yet')
    revision_file.write_text(json.dumps(revisions, indent=2))
os.chdir(REPO_DIR)
print(json.dumps(revisions, indent=2))

## Clone đúng repo dữ liệu, đọc duy nhất folder final

In [ ]:
DATA_REPO = Path('/content/Prepare-data-HotpotQA-VN')
if not (DATA_REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/chichic21039/Prepare-data-HotpotQA-VN.git', str(DATA_REPO)], check=True)
dataset_revision_file = RUN_ROOT / 'dataset_revision.txt'
if dataset_revision_file.exists():
    commit = dataset_revision_file.read_text().strip()
    status = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=DATA_REPO, text=True).strip()
    if status:
        raise RuntimeError('Dataset clone has local edits; preserve them before switching revision')
    subprocess.run(['git', 'checkout', '--detach', commit], cwd=DATA_REPO, check=True)
else:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=DATA_REPO, text=True).strip()
    dataset_revision_file.write_text(commit)
FINAL_DIR = DATA_REPO / 'data/hotpotqa_vi_1k/final'
print('Dataset revision:', commit)
print('Reading:', FINAL_DIR)


## Môi trường tách biệt

QA/KG client dùng CPU; vLLM chỉ được cài cho phase cần LLM/GPU. Lần đầu có thể tải nhiều thư viện. Mỗi bước hiện log trực tiếp và lưu log trên Drive; lock được dùng lại khi reconnect.

In [ ]:
from collections import deque
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True)
subprocess.run(['uv', '--version'], check=True)
QA_ENV = Path('/content/autoschema_qa_env')
VLLM_ENV = Path('/content/autoschema_vllm_env')
QA_PY = str(QA_ENV / 'bin/python')
VLLM_PY = str(VLLM_ENV / 'bin/python')
QA_REQUIREMENTS = REPO_DIR / 'requirements-hotpotqa-v2.txt'
COLAB_REQUIREMENTS = REPO_DIR / 'requirements-colab.txt'
required_setup_files = [QA_REQUIREMENTS, COLAB_REQUIREMENTS, REPO_DIR / 'pyproject.toml']
missing_setup_files = [path for path in required_setup_files if not path.is_file()]
if missing_setup_files:
    raise FileNotFoundError('Missing setup files: ' + ', '.join(map(str, missing_setup_files)))

def run_logged(label, command):
    log_path = RUN_ROOT / f'{label}.log'
    recent = deque(maxlen=60)
    print(f'\n=== {label} ===', flush=True)
    print('Command:', ' '.join(map(str, command)), flush=True)
    print('Log:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(list(map(str, command)), cwd=str(REPO_DIR),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
            recent.append(line.rstrip())
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with code {return_code}. Full log: {log_path}\n'
                           + '\n'.join(recent))

if not Path(QA_PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', str(QA_ENV)], check=True)
qa_lock = RUN_ROOT / 'qa_requirements.lock.txt'
if qa_lock.exists() and qa_lock.stat().st_size:
    run_logged('qa_lock_install', ['uv', 'pip', 'install', '--python', QA_PY,
        '--torch-backend=cpu', '-r', qa_lock])
else:
    run_logged('qa_base_install', ['uv', 'pip', 'install', '--python', QA_PY,
        '--torch-backend=cpu', '-r', QA_REQUIREMENTS])
    run_logged('qa_repo_install', ['uv', 'pip', 'install', '--python', QA_PY,
        '--torch-backend=cpu', '--constraint', QA_REQUIREMENTS, '-r', COLAB_REQUIREMENTS])
    with qa_lock.open('w', encoding='utf-8') as lock_file:
        subprocess.run(['uv', 'pip', 'freeze', '--python', QA_PY], cwd=str(REPO_DIR),
            stdout=lock_file, text=True, check=True)

if NEEDS_LLM:
    if not Path(VLLM_PY).exists():
        subprocess.run(['uv', 'venv', '--python', '3.12', str(VLLM_ENV)], check=True)
    vllm_lock = RUN_ROOT / 'vllm_requirements.lock.txt'
    vllm_args = ['-r', vllm_lock] if vllm_lock.exists() and vllm_lock.stat().st_size else ['--pre', 'vllm']
    run_logged('vllm_install', ['uv', 'pip', 'install', '--python', VLLM_PY,
        '--torch-backend=auto', *vllm_args])
    if not vllm_lock.exists() or not vllm_lock.stat().st_size:
        with vllm_lock.open('w', encoding='utf-8') as lock_file:
            subprocess.run(['uv', 'pip', 'freeze', '--python', VLLM_PY], cwd=str(REPO_DIR),
                stdout=lock_file, text=True, check=True)
else:
    print('Skipping vLLM installation for CPU-only phase:', RUN_PHASE)

subprocess.run([QA_PY, '-c', "import torch; print('QA torch:', torch.__version__)"], check=True)


## Khởi động Qwen nếu giai đoạn cần LLM

In [ ]:
if NEEDS_LLM:
    LOG_PATH = RUN_ROOT / 'qwen_vllm.log'
    def ready():
        try:
            response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
            if response.ok:
                models = response.json()['data']
                if not any(m['id'] == MODEL_ID for m in models):
                    raise RuntimeError('Port is occupied by another model; change PORT')
                return True
        except requests.RequestException:
            return False
        return False
    if not ready():
        server_log = open(LOG_PATH, 'a', encoding='utf-8')
        server = subprocess.Popen([str(VLLM_ENV / 'bin/vllm'), 'serve', MODEL_ID,
            '--revision', revisions['model_revision'], '--host', '127.0.0.1', '--port', str(PORT),
            '--dtype', 'half', '--max-model-len', str(CONTEXT_LENGTH), '--max-num-seqs', '1',
            '--gpu-memory-utilization', '0.80', '--language-model-only'],
            stdout=server_log, stderr=subprocess.STDOUT)
        started = time.monotonic()
        while time.monotonic() - started < 1200:
            if server.poll() is not None:
                server_log.flush()
                print(LOG_PATH.read_text(errors='replace')[-12000:])
                raise RuntimeError('vLLM stopped; inspect log above')
            if ready():
                break
            print(f'Waiting for Qwen: {time.monotonic() - started:.0f}s', flush=True)
            time.sleep(10)
        else:
            raise TimeoutError(f'Server not ready. Inspect {LOG_PATH}; do not start another copy')
    print('Local Qwen is ready')
else:
    print('This phase does not require Qwen.')


## Chạy giai đoạn đã chọn

`prepare`: không gọi LLM. `extract`: ba bước trích xuất trên toàn corpus và flush checkpoint sau từng chunk. `EXPERIMENT_PROFILE` quyết định thư mục, repetition penalty và số chunk mới của mỗi lượt; pilot mặc định xử lý đúng 109 chunk. Chạy lại `extract` không đổi profile để resume. Nếu Colab disconnect, dùng lại đúng `RUN_ROOT`: wrapper kiểm tra JSONL rồi tự bỏ qua các chunk đã hoàn tất. `build`: sinh concept và GraphML. `package`: ZIP graph/provenance. `benchmark`: 4 cấu hình, checkpoint từng câu. Không dùng `--overwrite`.

In [ ]:
RUN_SCRIPT = REPO_DIR / 'scripts' / 'run_hotpotqa_vn.py'
if not RUN_SCRIPT.is_file():
    raise FileNotFoundError(f'Missing runner: {RUN_SCRIPT}. Check CODE_REF and rerun the clone cell.')
required_vn_files = [FINAL_DIR / name for name in ('queries.jsonl', 'corpus.jsonl', 'qrels.tsv')]
missing_vn_files = [path for path in required_vn_files if not path.is_file()]
if missing_vn_files:
    raise FileNotFoundError('Missing HotpotQA-VN final files: ' + ', '.join(map(str, missing_vn_files)))

command = [QA_PY, '-X', 'utf8', '-u', str(RUN_SCRIPT),
    '--phase', RUN_PHASE, '--source-dir', str(FINAL_DIR), '--work-dir', str(WORK_DIR),
    '--max-questions', str(MAX_QUESTIONS), '--sampling', 'random', '--seed', '42']

# prepare/package do not use either model. Add model arguments only to phases that need them.
if RUN_PHASE in {'extract', 'build', 'benchmark', 'all'}:
    command += ['--model', MODEL_ID, '--model-revision', revisions['model_revision'],
        '--base-url', f'http://127.0.0.1:{PORT}/v1', '--context-length', str(CONTEXT_LENGTH)]
if RUN_PHASE in {'extract', 'all'}:
    command += ['--max-extraction-chunks', str(EXTRACTION_CHUNKS_PER_RUN),
        '--repetition-penalty', str(REPETITION_PENALTY)]
if RUN_PHASE in {'extract', 'build', 'all'} and not INCLUDE_EVENTS:
    command.append('--without-events')
elif RUN_PHASE in {'extract', 'build', 'all'} and not INCLUDE_EVENT_RELATIONS:
    command.append('--without-event-relations')
if RUN_PHASE in {'build'} and ALLOW_PARTIAL_BUILD:
    command.append('--allow-partial-build')
if RUN_PHASE in {'benchmark', 'all'}:
    command += ['--embedding-model', EMBEDDING_MODEL,
        '--embedding-revision', revisions['embedding_revision'],
        '--top-passages', '10', '--variants', BENCHMARK_VARIANT]

if RUN_PHASE in {'extract', 'all'}:
    for option in ('--max-extraction-chunks', '--repetition-penalty'):
        if command.count(option) != 1:
            raise RuntimeError(f'{option} must occur exactly once; do not append a second extraction block')
if any('[' in str(value) or '](' in str(value) for value in command):
    raise ValueError('Command contains a Markdown-formatted URL; use http://127.0.0.1:8000/v1')

phase_log = RUN_ROOT / f'{RUN_PHASE}.log'
print('Running:', ' '.join(map(str, command)), flush=True)
print('Log:', phase_log, flush=True)
with phase_log.open('a', encoding='utf-8') as log_file:
    process = subprocess.Popen(command, cwd=str(REPO_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding='utf-8', errors='replace', bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Phase {RUN_PHASE} failed with code {return_code}. Full log: {phase_log}')
progress_file = WORK_DIR / 'graph' / 'extraction_progress.json'
if progress_file.exists():
    print('Extraction progress:', progress_file.read_text(encoding='utf-8'), flush=True)
extraction_marker = WORK_DIR / 'graph' / 'vn_extraction_complete.json'
if RUN_PHASE in {'extract', 'all'} and not extraction_marker.exists():
    print('Checkpoint saved; rerun RUN_PHASE=\'extract\' unchanged to continue.', flush=True)
else:
    print(f'Phase {RUN_PHASE} completed.', flush=True)


## Audit pilot 109 chunk

Chạy sau `extract`. Cell chỉ đọc checkpoint/log, tạo corrected audit, duplicate concentration, 20 mẫu review và so sánh với pilot RP 1.15. Không cần gọi LLM.

In [ ]:
progress_file = WORK_DIR / 'graph' / 'extraction_progress.json'
if not progress_file.is_file():
    print('Run extract before audit.')
else:
    progress = json.loads(progress_file.read_text(encoding='utf-8'))
    completed = progress.get('completed_chunks', 0)
    if completed < 109:
        print(f'Pilot incomplete: {completed}/109 chunks')
    else:
        baseline_root = Path('/content/drive/MyDrive/AutoSchemaKG/hotpotqa_vn_ab_rp115')
        audit_command = [QA_PY, '-X', 'utf8', '-u',
            str(REPO_DIR / 'scripts/audit_vn_extraction_pilot.py'), str(RUN_ROOT),
            '--baseline-root', str(baseline_root), '--pilot-chunks', '109',
            '--manual-size', '20', '--seed', '42']
        subprocess.run(audit_command, cwd=str(REPO_DIR), check=True)


## Lưu kết quả

Drive giữ input, graph, checkpoint, model/code/data revisions và dependency locks. Số liệu chỉ là kết quả hoàn chỉnh khi mọi method có `complete=true`.

In [ ]:
if (OUTPUT_DIR / 'summary.json').exists():
    print((OUTPUT_DIR / 'summary.json').read_text())
print('Saved experiment:', WORK_DIR)
# Results already persist in Drive; no automatic ZIP or browser download.
print('Saved experiment:', RUN_ROOT)
print("Optional download: archive = shutil.make_archive('/content/autoschemakg_hotpotqa_vn', 'zip', RUN_ROOT); files.download(archive)")
